In [1]:
# Model N1 v2: Random forest, no winsorization
# Hyperparameters tuned by six-fold expanding-window CV through 2021
# Thesis-standard implementation under final horse-race plan v2

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

# =========================================================
# 1. USER SETTINGS
# =========================================================
BASE_DIR = Path(".")

PANEL_PATH = BASE_DIR / "Master_Modeling_Panel_v2.csv"

TUNING_SUMMARY_CSV = BASE_DIR / "Model_N1_RF_NoWinsor_v1_Tuning_Summary.csv"
CV_FOLD_METRICS_CSV = BASE_DIR / "Model_N1_RF_NoWinsor_v1_SelectedHP_DevCV_Fold_Metrics.csv"
STAGE_METRICS_CSV = BASE_DIR / "Model_N1_RF_NoWinsor_v1_Stage_Metrics.csv"
PREDICTIONS_CSV = BASE_DIR / "Model_N1_RF_NoWinsor_v1_Predictions.csv"

# Candidate hyperparameter grids from Final_Model_Horse_Race_Plan_v2.xlsx
N_ESTIMATORS_GRID = [300, 500, 800]
MAX_DEPTH_GRID = [3, 5, 7, None]
MIN_SAMPLES_LEAF_GRID = [5, 20, 50]
MAX_FEATURES_GRID = ["sqrt", 0.33, 0.5]

# Fix randomness because random forests are stochastic
RF_RANDOM_STATE = 42

# =========================================================
# 2. LOAD FROZEN PANEL
# =========================================================
df = pd.read_csv(PANEL_PATH, low_memory=False)

# =========================================================
# 3. LOCKED TARGET + SPLITS
# =========================================================
TARGET_COL = "targeted_in_year"

def assign_split(year: int) -> str:
    if 2010 <= year <= 2021:
        return "development"
    if 2022 <= year <= 2024:
        return "final_test"
    return "exclude"

df["split"] = df["year"].apply(assign_split)
df = df[df["split"] != "exclude"].copy()

# =========================================================
# 4. LOCKED RAW PREDICTOR SET (SELF-CONTAINED, MATCHES LOCKED SPEC)
# =========================================================
continuous_features = [
    "roe",
    "assets_to_equity",
    "current_ratio",
    "ebitda",
    "ebitda_margin",
    "sales_to_assets",
    "sales_growth",
    "interest_coverage",
    "net_debt_to_ebitda",
    "fcf_to_capex",
    "market_cap",
    "ret_3m",
    "ret_6m",
    "ret_1y",
    "ret_2y",
    "ret_3y",
    "ret_5y",
    "volatility_30d",
    "volatility_90d",
    "volatility_180d",
    "turnover_30d",
    "dividend_payout_ratio",
    "buyback_yield",
    "price_to_book",
    "ev_to_sales",
    "ev_to_ebitda",
    "tobins_q",
    "fcf_yield",
    "prior_campaign_count_3y",
    "prior_settlement_count_3y",
    "prior_management_favorable_count_3y",
    "prior_dissident_favorable_count_3y",
    "prior_campaign_count_5y",
    "prior_settlement_count_5y",
    "prior_management_favorable_count_5y",
    "prior_dissident_favorable_count_5y",
    "ff49_other_permno_years_in_category",
    "ret_3m_rel_peer",
    "ret_6m_rel_peer",
    "ret_1y_rel_peer",
    "ret_2y_rel_peer",
    "ret_3y_rel_peer",
    "ret_5y_rel_peer",
    "log_ev_to_sales_rel_peer",
    "log_price_to_book_rel_peer",
    "log_tobins_q_rel_peer",
    "log_ev_to_ebitda_rel_peer",
    "ebitda_margin_rel_peer",
    "sales_growth_rel_peer",
    "roe_rel_peer",
    "board_size",
    "board_female_ratio",
    "ceo_tenure",
    "top10_inst_conc_within_13f",
    "inst_ownership_pct_13f",
]

binary_features = [
    "prior_activism_3y",
    "prior_activism_5y",
    "history_3y_observed",
    "history_5y_observed",
    "ff49_unclassified",
    "classified_board",
    "poison_pill",
    "dual_class",
    "iss_match_found",
    "rm_stale_gt_730",
    "board_match_found",
    "board_stale_gt_365",
    "ceo_is_female",
    "ceo_chair_duality",
    "ceo_match_found",
    "ceo_stale_gt_365",
    "inst_match_found",
]

categorical_feature = "ff17_for_model"
df[categorical_feature] = df["ff17_short"].fillna("Unclassified").astype(str)

ff17_categories = [
    "Cars",
    "Chems",
    "Clths",
    "Cnstr",
    "Cnsum",
    "Durbl",
    "FabPr",
    "Finan",
    "Food",
    "Machn",
    "Mines",
    "Oil",
    "Other",          # omitted reference
    "Rtail",
    "Steel",
    "Trans",
    "Utils",
    "Unclassified",
]

required_cols = continuous_features + binary_features + [categorical_feature, TARGET_COL, "year", "permno"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

all_predictors = continuous_features + binary_features + [categorical_feature]

# =========================================================
# 5. THESIS-STANDARD METRICS
# =========================================================
def evaluate_predictions(y_true: pd.Series, y_prob: np.ndarray) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    return {
        "pr_auc": average_precision_score(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "brier_score": brier_score_loss(y_true, y_prob),
    }

# =========================================================
# 6. PREPROCESSOR + MODEL PIPELINE BUILDER
# =========================================================
def build_pipeline(
    n_estimators: int,
    max_depth,
    min_samples_leaf: int,
    max_features,
):
    continuous_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )

    binary_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unclassified")),
            (
                "onehot",
                OneHotEncoder(
                    categories=[ff17_categories],
                    drop=["Other"],           # locked reference category
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("cont", continuous_transformer, continuous_features),
            ("bin", binary_transformer, binary_features),
            ("ff17", categorical_transformer, [categorical_feature]),
        ],
        remainder="drop",
    )

    rf_model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        bootstrap=True,
        n_jobs=-1,
        random_state=RF_RANDOM_STATE,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", rf_model),
        ]
    )

# =========================================================
# 7. FINAL SIX-FOLD EXPANDING-WINDOW CV
# =========================================================
cv_fold_defs = [
    {"fold": "fold_1", "train_years": list(range(2010, 2016)), "val_years": [2016]},
    {"fold": "fold_2", "train_years": list(range(2010, 2017)), "val_years": [2017]},
    {"fold": "fold_3", "train_years": list(range(2010, 2018)), "val_years": [2018]},
    {"fold": "fold_4", "train_years": list(range(2010, 2019)), "val_years": [2019]},
    {"fold": "fold_5", "train_years": list(range(2010, 2020)), "val_years": [2020]},
    {"fold": "fold_6", "train_years": list(range(2010, 2021)), "val_years": [2021]},
]

# =========================================================
# 8. GRID SEARCH OVER EXPANDING FOLDS
# =========================================================
tuning_rows = []

for n_estimators in N_ESTIMATORS_GRID:
    for max_depth in MAX_DEPTH_GRID:
        for min_samples_leaf in MIN_SAMPLES_LEAF_GRID:
            for max_features in MAX_FEATURES_GRID:
                fold_rows = []

                for fold_def in cv_fold_defs:
                    fold_name = fold_def["fold"]
                    train_years = fold_def["train_years"]
                    val_years = fold_def["val_years"]

                    train_df = df[df["year"].isin(train_years)].copy()
                    val_df = df[df["year"].isin(val_years)].copy()

                    X_train = train_df[all_predictors].copy()
                    y_train = train_df[TARGET_COL].copy()

                    X_val = val_df[all_predictors].copy()
                    y_val = val_df[TARGET_COL].copy()

                    pipeline = build_pipeline(
                        n_estimators=n_estimators,
                        max_depth=max_depth,
                        min_samples_leaf=min_samples_leaf,
                        max_features=max_features,
                    )
                    pipeline.fit(X_train, y_train)
                    val_prob = pipeline.predict_proba(X_val)[:, 1]

                    fold_metrics = evaluate_predictions(y_val, val_prob)
                    fold_metrics.update(
                        {
                            "fold": fold_name,
                            "train_year_start": min(train_years),
                            "train_year_end": max(train_years),
                            "val_year_start": min(val_years),
                            "val_year_end": max(val_years),
                            "train_rows": len(train_df),
                            "train_positives": int(y_train.sum()),
                            "train_positive_rate": float(y_train.mean()),
                            "val_rows": len(val_df),
                            "val_positives": int(y_val.sum()),
                            "val_positive_rate": float(y_val.mean()),
                            "mean_predicted_probability": float(np.mean(val_prob)),
                        }
                    )
                    fold_rows.append(fold_metrics)

                fold_metrics_df = pd.DataFrame(fold_rows)

                tuning_rows.append(
                    {
                        "n_estimators": n_estimators,
                        "max_depth": max_depth,
                        "min_samples_leaf": min_samples_leaf,
                        "max_features": max_features,
                        "cv_mean_pr_auc": float(fold_metrics_df["pr_auc"].mean()),
                        "cv_std_pr_auc": float(fold_metrics_df["pr_auc"].std(ddof=1)),
                        "cv_mean_roc_auc": float(fold_metrics_df["roc_auc"].mean()),
                        "cv_std_roc_auc": float(fold_metrics_df["roc_auc"].std(ddof=1)),
                        "cv_mean_brier_score": float(fold_metrics_df["brier_score"].mean()),
                        "cv_std_brier_score": float(fold_metrics_df["brier_score"].std(ddof=1)),
                    }
                )

tuning_summary_df = pd.DataFrame(tuning_rows)

# Preference ordering:
# 1) higher mean CV PR-AUC
# 2) higher mean CV ROC-AUC
# 3) lower mean CV Brier
# 4) simpler forest as a practical tie-break if metrics are essentially tied
#    (shallower depth, larger leaves, fewer trees)
depth_rank_map = {3: 0, 5: 1, 7: 2, None: 3}
max_features_rank_map = {"sqrt": 0, 0.33: 1, 0.5: 2}

tuning_summary_df["depth_rank"] = tuning_summary_df["max_depth"].map(depth_rank_map)
tuning_summary_df["max_features_rank"] = tuning_summary_df["max_features"].map(max_features_rank_map)

tuning_summary_df = tuning_summary_df.sort_values(
    by=[
        "cv_mean_pr_auc",
        "cv_mean_roc_auc",
        "cv_mean_brier_score",
        "depth_rank",
        "min_samples_leaf",
        "n_estimators",
        "max_features_rank",
    ],
    ascending=[False, False, True, True, False, True, True],
).reset_index(drop=True)

tuning_summary_df.to_csv(TUNING_SUMMARY_CSV, index=False)

selected_n_estimators = int(tuning_summary_df.loc[0, "n_estimators"])
selected_max_depth = tuning_summary_df.loc[0, "max_depth"]
if pd.isna(selected_max_depth):
    selected_max_depth = None
else:
    selected_max_depth = int(selected_max_depth)

selected_min_samples_leaf = int(tuning_summary_df.loc[0, "min_samples_leaf"])
selected_max_features = tuning_summary_df.loc[0, "max_features"]

# Restore numeric type for max_features when applicable
if isinstance(selected_max_features, str):
    if selected_max_features != "sqrt":
        selected_max_features = float(selected_max_features)

# =========================================================
# 9. RERUN SELECTED HYPERPARAMETERS FOR FOLD METRICS
# =========================================================
selected_cv_rows = []
prediction_frames = []

for fold_def in cv_fold_defs:
    fold_name = fold_def["fold"]
    train_years = fold_def["train_years"]
    val_years = fold_def["val_years"]

    train_df = df[df["year"].isin(train_years)].copy()
    val_df = df[df["year"].isin(val_years)].copy()

    X_train = train_df[all_predictors].copy()
    y_train = train_df[TARGET_COL].copy()

    X_val = val_df[all_predictors].copy()
    y_val = val_df[TARGET_COL].copy()

    pipeline = build_pipeline(
        n_estimators=selected_n_estimators,
        max_depth=selected_max_depth,
        min_samples_leaf=selected_min_samples_leaf,
        max_features=selected_max_features,
    )
    pipeline.fit(X_train, y_train)
    val_prob = pipeline.predict_proba(X_val)[:, 1]

    fold_metrics = evaluate_predictions(y_val, val_prob)
    fold_metrics.update(
        {
            "selected_n_estimators": selected_n_estimators,
            "selected_max_depth": selected_max_depth,
            "selected_min_samples_leaf": selected_min_samples_leaf,
            "selected_max_features": selected_max_features,
            "fold": fold_name,
            "train_year_start": min(train_years),
            "train_year_end": max(train_years),
            "val_year_start": min(val_years),
            "val_year_end": max(val_years),
            "train_rows": len(train_df),
            "train_positives": int(y_train.sum()),
            "train_positive_rate": float(y_train.mean()),
            "val_rows": len(val_df),
            "val_positives": int(y_val.sum()),
            "val_positive_rate": float(y_val.mean()),
            "mean_predicted_probability": float(np.mean(val_prob)),
        }
    )
    selected_cv_rows.append(fold_metrics)

    prediction_frames.append(
        pd.DataFrame(
            {
                "permno": val_df["permno"].to_numpy(),
                "year": val_df["year"].to_numpy(),
                "evaluation_stage": "development_cv",
                "fold": fold_name,
                "actual": y_val.to_numpy(),
                "predicted_probability": val_prob,
                "selected_n_estimators": selected_n_estimators,
                "selected_max_depth": selected_max_depth,
                "selected_min_samples_leaf": selected_min_samples_leaf,
                "selected_max_features": selected_max_features,
            }
        )
    )

selected_cv_df = pd.DataFrame(selected_cv_rows)
selected_cv_df.to_csv(CV_FOLD_METRICS_CSV, index=False)

# =========================================================
# 10. FINAL LOCKED TEST: REFIT ON 2010-2021, TEST ON 2022-2024
# =========================================================
dev_df = df[df["year"].between(2010, 2021)].copy()
test_df = df[df["year"].between(2022, 2024)].copy()

X_dev = dev_df[all_predictors].copy()
y_dev = dev_df[TARGET_COL].copy()

X_test = test_df[all_predictors].copy()
y_test = test_df[TARGET_COL].copy()

final_pipeline = build_pipeline(
    n_estimators=selected_n_estimators,
    max_depth=selected_max_depth,
    min_samples_leaf=selected_min_samples_leaf,
    max_features=selected_max_features,
)
final_pipeline.fit(X_dev, y_dev)

test_prob = final_pipeline.predict_proba(X_test)[:, 1]
test_metrics = evaluate_predictions(y_test, test_prob)

prediction_frames.append(
    pd.DataFrame(
        {
            "permno": test_df["permno"].to_numpy(),
            "year": test_df["year"].to_numpy(),
            "evaluation_stage": "final_test",
            "fold": "final_test",
            "actual": y_test.to_numpy(),
            "predicted_probability": test_prob,
            "selected_n_estimators": selected_n_estimators,
            "selected_max_depth": selected_max_depth,
            "selected_min_samples_leaf": selected_min_samples_leaf,
            "selected_max_features": selected_max_features,
        }
    )
)

predictions_df = pd.concat(prediction_frames, ignore_index=True)
predictions_df.to_csv(PREDICTIONS_CSV, index=False)

# =========================================================
# 11. STAGE METRICS TABLE
# =========================================================
stage_metrics_df = pd.DataFrame(
    [
        {
            "stage": "development_cv_mean",
            "selected_n_estimators": selected_n_estimators,
            "selected_max_depth": selected_max_depth,
            "selected_min_samples_leaf": selected_min_samples_leaf,
            "selected_max_features": selected_max_features,
            "pr_auc": float(selected_cv_df["pr_auc"].mean()),
            "roc_auc": float(selected_cv_df["roc_auc"].mean()),
            "brier_score": float(selected_cv_df["brier_score"].mean()),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
        },
        {
            "stage": "development_cv_std",
            "selected_n_estimators": selected_n_estimators,
            "selected_max_depth": selected_max_depth,
            "selected_min_samples_leaf": selected_min_samples_leaf,
            "selected_max_features": selected_max_features,
            "pr_auc": float(selected_cv_df["pr_auc"].std(ddof=1)),
            "roc_auc": float(selected_cv_df["roc_auc"].std(ddof=1)),
            "brier_score": float(selected_cv_df["brier_score"].std(ddof=1)),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
        },
        {
            "stage": "final_test",
            "selected_n_estimators": selected_n_estimators,
            "selected_max_depth": selected_max_depth,
            "selected_min_samples_leaf": selected_min_samples_leaf,
            "selected_max_features": selected_max_features,
            "pr_auc": float(test_metrics["pr_auc"]),
            "roc_auc": float(test_metrics["roc_auc"]),
            "brier_score": float(test_metrics["brier_score"]),
            "rows": int(len(test_df)),
            "positives": int(y_test.sum()),
            "positive_rate": float(y_test.mean()),
            "mean_predicted_probability": float(np.mean(test_prob)),
        },
    ]
)

stage_metrics_df.to_csv(STAGE_METRICS_CSV, index=False)

print("N1 random forest run complete.")
print(f"Saved tuning summary to: {TUNING_SUMMARY_CSV}")
print(f"Saved selected-HP fold metrics to: {CV_FOLD_METRICS_CSV}")
print(f"Saved stage metrics to: {STAGE_METRICS_CSV}")
print(f"Saved predictions to: {PREDICTIONS_CSV}")
print()
print("Selected hyperparameters:")
print(f"  n_estimators      = {selected_n_estimators}")
print(f"  max_depth         = {selected_max_depth}")
print(f"  min_samples_leaf  = {selected_min_samples_leaf}")
print(f"  max_features      = {selected_max_features}")

N1 random forest run complete.
Saved tuning summary to: Model_N1_RF_NoWinsor_v2_Tuning_Summary.csv
Saved selected-HP fold metrics to: Model_N1_RF_NoWinsor_v2_SelectedHP_DevCV_Fold_Metrics.csv
Saved stage metrics to: Model_N1_RF_NoWinsor_v2_Stage_Metrics.csv
Saved predictions to: Model_N1_RF_NoWinsor_v2_Predictions.csv

Selected hyperparameters:
  n_estimators      = 500
  max_depth         = None
  min_samples_leaf  = 50
  max_features      = 0.5


In [6]:
# Model N2 v2: Random forest, with winsorization
# Hyperparameters tuned by six-fold expanding-window CV through 2021
# Thesis-standard implementation under final horse-race plan v2

import pandas as pd
import numpy as np
from pathlib import Path
import time
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

# =========================================================
# 1. USER SETTINGS
# =========================================================
BASE_DIR = Path(".")

PANEL_PATH = BASE_DIR / "Master_Modeling_Panel_v2.csv"

TUNING_SUMMARY_CSV = BASE_DIR / "Model_N2_RF_Winsor_v1_Tuning_Summary.csv"
CV_FOLD_METRICS_CSV = BASE_DIR / "Model_N2_RF_Winsor_v1_SelectedHP_DevCV_Fold_Metrics.csv"
STAGE_METRICS_CSV = BASE_DIR / "Model_N2_RF_Winsor_v1_Stage_Metrics.csv"
PREDICTIONS_CSV = BASE_DIR / "Model_N2_RF_Winsor_v1_Predictions.csv"

# Candidate hyperparameter grids from Final_Model_Horse_Race_Plan_v2.xlsx
N_ESTIMATORS_GRID = [300, 500, 800]
MAX_DEPTH_GRID = [3, 5, 7, None]
MIN_SAMPLES_LEAF_GRID = [5, 20, 50]
MAX_FEATURES_GRID = ["sqrt", 0.33, 0.5]

# Fix randomness because random forests are stochastic
RF_RANDOM_STATE = 42

# =========================================================
# 2. LOAD FROZEN PANEL
# =========================================================
df = pd.read_csv(PANEL_PATH, low_memory=False)

# =========================================================
# 3. LOCKED TARGET + SPLITS
# =========================================================
TARGET_COL = "targeted_in_year"

def assign_split(year: int) -> str:
    if 2010 <= year <= 2021:
        return "development"
    if 2022 <= year <= 2024:
        return "final_test"
    return "exclude"

df["split"] = df["year"].apply(assign_split)
df = df[df["split"] != "exclude"].copy()

# =========================================================
# 4. LOCKED RAW PREDICTOR SET (SELF-CONTAINED, MATCHES LOCKED SPEC)
# =========================================================
continuous_features = [
    "roe",
    "assets_to_equity",
    "current_ratio",
    "ebitda",
    "ebitda_margin",
    "sales_to_assets",
    "sales_growth",
    "interest_coverage",
    "net_debt_to_ebitda",
    "fcf_to_capex",
    "market_cap",
    "ret_3m",
    "ret_6m",
    "ret_1y",
    "ret_2y",
    "ret_3y",
    "ret_5y",
    "volatility_30d",
    "volatility_90d",
    "volatility_180d",
    "turnover_30d",
    "dividend_payout_ratio",
    "buyback_yield",
    "price_to_book",
    "ev_to_sales",
    "ev_to_ebitda",
    "tobins_q",
    "fcf_yield",
    "prior_campaign_count_3y",
    "prior_settlement_count_3y",
    "prior_management_favorable_count_3y",
    "prior_dissident_favorable_count_3y",
    "prior_campaign_count_5y",
    "prior_settlement_count_5y",
    "prior_management_favorable_count_5y",
    "prior_dissident_favorable_count_5y",
    "ff49_other_permno_years_in_category",
    "ret_3m_rel_peer",
    "ret_6m_rel_peer",
    "ret_1y_rel_peer",
    "ret_2y_rel_peer",
    "ret_3y_rel_peer",
    "ret_5y_rel_peer",
    "log_ev_to_sales_rel_peer",
    "log_price_to_book_rel_peer",
    "log_tobins_q_rel_peer",
    "log_ev_to_ebitda_rel_peer",
    "ebitda_margin_rel_peer",
    "sales_growth_rel_peer",
    "roe_rel_peer",
    "board_size",
    "board_female_ratio",
    "ceo_tenure",
    "top10_inst_conc_within_13f",
    "inst_ownership_pct_13f",
]

binary_features = [
    "prior_activism_3y",
    "prior_activism_5y",
    "history_3y_observed",
    "history_5y_observed",
    "ff49_unclassified",
    "classified_board",
    "poison_pill",
    "dual_class",
    "iss_match_found",
    "rm_stale_gt_730",
    "board_match_found",
    "board_stale_gt_365",
    "ceo_is_female",
    "ceo_chair_duality",
    "ceo_match_found",
    "ceo_stale_gt_365",
    "inst_match_found",
]

categorical_feature = "ff17_for_model"
df[categorical_feature] = df["ff17_short"].fillna("Unclassified").astype(str)

ff17_categories = [
    "Cars",
    "Chems",
    "Clths",
    "Cnstr",
    "Cnsum",
    "Durbl",
    "FabPr",
    "Finan",
    "Food",
    "Machn",
    "Mines",
    "Oil",
    "Other",          # omitted reference
    "Rtail",
    "Steel",
    "Trans",
    "Utils",
    "Unclassified",
]

required_cols = continuous_features + binary_features + [categorical_feature, TARGET_COL, "year", "permno"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

all_predictors = continuous_features + binary_features + [categorical_feature]

# =========================================================
# 5. THESIS-STANDARD METRICS
# =========================================================
def evaluate_predictions(y_true: pd.Series, y_prob: np.ndarray) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    return {
        "pr_auc": average_precision_score(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "brier_score": brier_score_loss(y_true, y_prob),
    }

# =========================================================
# 6. TRAIN-ONLY WINSORIZER
# =========================================================
class PercentileWinsorizer(BaseEstimator, TransformerMixin):
    """
    Column-wise winsorization using training-sample percentiles only.
    NaNs are ignored when fitting percentiles and preserved until imputation.
    """
    def __init__(self, lower=1.0, upper=99.0):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        n_cols = X.shape[1]
        lower_bounds = []
        upper_bounds = []

        for j in range(n_cols):
            col = X[:, j]
            non_missing = col[~np.isnan(col)]
            if non_missing.size == 0:
                lower_bounds.append(np.nan)
                upper_bounds.append(np.nan)
            else:
                lower_bounds.append(np.percentile(non_missing, self.lower))
                upper_bounds.append(np.percentile(non_missing, self.upper))

        self.lower_bounds_ = np.array(lower_bounds, dtype=float)
        self.upper_bounds_ = np.array(upper_bounds, dtype=float)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float).copy()
        for j in range(X.shape[1]):
            lb = self.lower_bounds_[j]
            ub = self.upper_bounds_[j]
            if np.isnan(lb) or np.isnan(ub):
                continue
            mask = ~np.isnan(X[:, j])
            X[mask, j] = np.clip(X[mask, j], lb, ub)
        return X

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)

# =========================================================
# 7. PREPROCESSOR + MODEL PIPELINE BUILDER
# =========================================================
def build_pipeline(
    n_estimators: int,
    max_depth,
    min_samples_leaf: int,
    max_features,
):
    continuous_transformer = Pipeline(
        steps=[
            ("winsorizer", PercentileWinsorizer(lower=1.0, upper=99.0)),
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )

    binary_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unclassified")),
            (
                "onehot",
                OneHotEncoder(
                    categories=[ff17_categories],
                    drop=["Other"],           # locked reference category
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("cont", continuous_transformer, continuous_features),
            ("bin", binary_transformer, binary_features),
            ("ff17", categorical_transformer, [categorical_feature]),
        ],
        remainder="drop",
    )

    rf_model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        bootstrap=True,
        n_jobs=-1,
        random_state=RF_RANDOM_STATE,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", rf_model),
        ]
    )

# =========================================================
# 8. FINAL SIX-FOLD EXPANDING-WINDOW CV
# =========================================================
cv_fold_defs = [
    {"fold": "fold_1", "train_years": list(range(2010, 2016)), "val_years": [2016]},
    {"fold": "fold_2", "train_years": list(range(2010, 2017)), "val_years": [2017]},
    {"fold": "fold_3", "train_years": list(range(2010, 2018)), "val_years": [2018]},
    {"fold": "fold_4", "train_years": list(range(2010, 2019)), "val_years": [2019]},
    {"fold": "fold_5", "train_years": list(range(2010, 2020)), "val_years": [2020]},
    {"fold": "fold_6", "train_years": list(range(2010, 2021)), "val_years": [2021]},
]

# =========================================================
# 9. GRID SEARCH OVER EXPANDING FOLDS
# =========================================================
tuning_rows = []

total_combos = (
    len(N_ESTIMATORS_GRID)
    * len(MAX_DEPTH_GRID)
    * len(MIN_SAMPLES_LEAF_GRID)
    * len(MAX_FEATURES_GRID)
)
combo_counter = 0
search_start_time = time.time()

print("Starting N2 random forest tuning search...")
print(f"Total hyperparameter combinations: {total_combos}")
print(f"Total fold fits in tuning stage: {total_combos * len(cv_fold_defs)}")
print()

for n_estimators in N_ESTIMATORS_GRID:
    for max_depth in MAX_DEPTH_GRID:
        for min_samples_leaf in MIN_SAMPLES_LEAF_GRID:
            for max_features in MAX_FEATURES_GRID:
                combo_counter += 1
                combo_start_time = time.time()
                elapsed_minutes = (combo_start_time - search_start_time) / 60.0
                print(
                    f"[TUNING] Combo {combo_counter}/{total_combos} | "
                    f"elapsed={elapsed_minutes:.1f} min | "
                    f"n_estimators={n_estimators}, max_depth={max_depth}, "
                    f"min_samples_leaf={min_samples_leaf}, max_features={max_features}"
                )

                fold_rows = []

                for fold_def in cv_fold_defs:
                    fold_name = fold_def["fold"]
                    train_years = fold_def["train_years"]
                    val_years = fold_def["val_years"]

                    train_df = df[df["year"].isin(train_years)].copy()
                    val_df = df[df["year"].isin(val_years)].copy()

                    X_train = train_df[all_predictors].copy()
                    y_train = train_df[TARGET_COL].copy()

                    X_val = val_df[all_predictors].copy()
                    y_val = val_df[TARGET_COL].copy()

                    pipeline = build_pipeline(
                        n_estimators=n_estimators,
                        max_depth=max_depth,
                        min_samples_leaf=min_samples_leaf,
                        max_features=max_features,
                    )
                    pipeline.fit(X_train, y_train)
                    val_prob = pipeline.predict_proba(X_val)[:, 1]

                    fold_metrics = evaluate_predictions(y_val, val_prob)
                    fold_metrics.update(
                        {
                            "fold": fold_name,
                            "train_year_start": min(train_years),
                            "train_year_end": max(train_years),
                            "val_year_start": min(val_years),
                            "val_year_end": max(val_years),
                            "train_rows": len(train_df),
                            "train_positives": int(y_train.sum()),
                            "train_positive_rate": float(y_train.mean()),
                            "val_rows": len(val_df),
                            "val_positives": int(y_val.sum()),
                            "val_positive_rate": float(y_val.mean()),
                            "mean_predicted_probability": float(np.mean(val_prob)),
                        }
                    )
                    fold_rows.append(fold_metrics)

                fold_metrics_df = pd.DataFrame(fold_rows)

                combo_minutes = (time.time() - combo_start_time) / 60.0
                combo_mean_pr_auc = float(fold_metrics_df["pr_auc"].mean())
                combo_mean_roc_auc = float(fold_metrics_df["roc_auc"].mean())
                combo_mean_brier = float(fold_metrics_df["brier_score"].mean())

                tuning_rows.append(
                    {
                        "n_estimators": n_estimators,
                        "max_depth": max_depth,
                        "min_samples_leaf": min_samples_leaf,
                        "max_features": max_features,
                        "cv_mean_pr_auc": combo_mean_pr_auc,
                        "cv_std_pr_auc": float(fold_metrics_df["pr_auc"].std(ddof=1)),
                        "cv_mean_roc_auc": combo_mean_roc_auc,
                        "cv_std_roc_auc": float(fold_metrics_df["roc_auc"].std(ddof=1)),
                        "cv_mean_brier_score": combo_mean_brier,
                        "cv_std_brier_score": float(fold_metrics_df["brier_score"].std(ddof=1)),
                    }
                )

                print(
                    f"[TUNING] Completed combo {combo_counter}/{total_combos} in {combo_minutes:.2f} min | "
                    f"cv_mean_pr_auc={combo_mean_pr_auc:.4f}, "
                    f"cv_mean_roc_auc={combo_mean_roc_auc:.4f}, "
                    f"cv_mean_brier={combo_mean_brier:.5f}"
                )
                print()

tuning_summary_df = pd.DataFrame(tuning_rows)

# Preference ordering:
# 1) higher mean CV PR-AUC
# 2) higher mean CV ROC-AUC
# 3) lower mean CV Brier
# 4) simpler forest as a practical tie-break if metrics are essentially tied
#    (shallower depth, larger leaves, fewer trees)
depth_rank_map = {3: 0, 5: 1, 7: 2, None: 3}
max_features_rank_map = {"sqrt": 0, 0.33: 1, 0.5: 2}

tuning_summary_df["depth_rank"] = tuning_summary_df["max_depth"].map(depth_rank_map)
tuning_summary_df["max_features_rank"] = tuning_summary_df["max_features"].map(max_features_rank_map)

tuning_summary_df = tuning_summary_df.sort_values(
    by=[
        "cv_mean_pr_auc",
        "cv_mean_roc_auc",
        "cv_mean_brier_score",
        "depth_rank",
        "min_samples_leaf",
        "n_estimators",
        "max_features_rank",
    ],
    ascending=[False, False, True, True, False, True, True],
).reset_index(drop=True)

tuning_summary_df.to_csv(TUNING_SUMMARY_CSV, index=False)

selected_n_estimators = int(tuning_summary_df.loc[0, "n_estimators"])
selected_max_depth = tuning_summary_df.loc[0, "max_depth"]
if pd.isna(selected_max_depth):
    selected_max_depth = None
else:
    selected_max_depth = int(selected_max_depth)

selected_min_samples_leaf = int(tuning_summary_df.loc[0, "min_samples_leaf"])
selected_max_features = tuning_summary_df.loc[0, "max_features"]

# Restore numeric type for max_features when applicable
if isinstance(selected_max_features, str):
    if selected_max_features != "sqrt":
        selected_max_features = float(selected_max_features)

# =========================================================
# 10. RERUN SELECTED HYPERPARAMETERS FOR FOLD METRICS
# =========================================================
selected_cv_rows = []
prediction_frames = []

for fold_def in cv_fold_defs:
    fold_name = fold_def["fold"]
    train_years = fold_def["train_years"]
    val_years = fold_def["val_years"]

    train_df = df[df["year"].isin(train_years)].copy()
    val_df = df[df["year"].isin(val_years)].copy()

    X_train = train_df[all_predictors].copy()
    y_train = train_df[TARGET_COL].copy()

    X_val = val_df[all_predictors].copy()
    y_val = val_df[TARGET_COL].copy()

    pipeline = build_pipeline(
        n_estimators=selected_n_estimators,
        max_depth=selected_max_depth,
        min_samples_leaf=selected_min_samples_leaf,
        max_features=selected_max_features,
    )
    pipeline.fit(X_train, y_train)
    val_prob = pipeline.predict_proba(X_val)[:, 1]

    fold_metrics = evaluate_predictions(y_val, val_prob)
    fold_metrics.update(
        {
            "selected_n_estimators": selected_n_estimators,
            "selected_max_depth": selected_max_depth,
            "selected_min_samples_leaf": selected_min_samples_leaf,
            "selected_max_features": selected_max_features,
            "fold": fold_name,
            "train_year_start": min(train_years),
            "train_year_end": max(train_years),
            "val_year_start": min(val_years),
            "val_year_end": max(val_years),
            "train_rows": len(train_df),
            "train_positives": int(y_train.sum()),
            "train_positive_rate": float(y_train.mean()),
            "val_rows": len(val_df),
            "val_positives": int(y_val.sum()),
            "val_positive_rate": float(y_val.mean()),
            "mean_predicted_probability": float(np.mean(val_prob)),
        }
    )
    selected_cv_rows.append(fold_metrics)

    prediction_frames.append(
        pd.DataFrame(
            {
                "permno": val_df["permno"].to_numpy(),
                "year": val_df["year"].to_numpy(),
                "evaluation_stage": "development_cv",
                "fold": fold_name,
                "actual": y_val.to_numpy(),
                "predicted_probability": val_prob,
                "selected_n_estimators": selected_n_estimators,
                "selected_max_depth": selected_max_depth,
                "selected_min_samples_leaf": selected_min_samples_leaf,
                "selected_max_features": selected_max_features,
            }
        )
    )

selected_cv_df = pd.DataFrame(selected_cv_rows)
selected_cv_df.to_csv(CV_FOLD_METRICS_CSV, index=False)

# =========================================================
# 11. FINAL LOCKED TEST: REFIT ON 2010-2021, TEST ON 2022-2024
# =========================================================
dev_df = df[df["year"].between(2010, 2021)].copy()
test_df = df[df["year"].between(2022, 2024)].copy()

X_dev = dev_df[all_predictors].copy()
y_dev = dev_df[TARGET_COL].copy()

X_test = test_df[all_predictors].copy()
y_test = test_df[TARGET_COL].copy()

final_pipeline = build_pipeline(
    n_estimators=selected_n_estimators,
    max_depth=selected_max_depth,
    min_samples_leaf=selected_min_samples_leaf,
    max_features=selected_max_features,
)
final_pipeline.fit(X_dev, y_dev)

test_prob = final_pipeline.predict_proba(X_test)[:, 1]
test_metrics = evaluate_predictions(y_test, test_prob)

prediction_frames.append(
    pd.DataFrame(
        {
            "permno": test_df["permno"].to_numpy(),
            "year": test_df["year"].to_numpy(),
            "evaluation_stage": "final_test",
            "fold": "final_test",
            "actual": y_test.to_numpy(),
            "predicted_probability": test_prob,
            "selected_n_estimators": selected_n_estimators,
            "selected_max_depth": selected_max_depth,
            "selected_min_samples_leaf": selected_min_samples_leaf,
            "selected_max_features": selected_max_features,
        }
    )
)

predictions_df = pd.concat(prediction_frames, ignore_index=True)
predictions_df.to_csv(PREDICTIONS_CSV, index=False)

# =========================================================
# 12. STAGE METRICS TABLE
# =========================================================
stage_metrics_df = pd.DataFrame(
    [
        {
            "stage": "development_cv_mean",
            "selected_n_estimators": selected_n_estimators,
            "selected_max_depth": selected_max_depth,
            "selected_min_samples_leaf": selected_min_samples_leaf,
            "selected_max_features": selected_max_features,
            "pr_auc": float(selected_cv_df["pr_auc"].mean()),
            "roc_auc": float(selected_cv_df["roc_auc"].mean()),
            "brier_score": float(selected_cv_df["brier_score"].mean()),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
        },
        {
            "stage": "development_cv_std",
            "selected_n_estimators": selected_n_estimators,
            "selected_max_depth": selected_max_depth,
            "selected_min_samples_leaf": selected_min_samples_leaf,
            "selected_max_features": selected_max_features,
            "pr_auc": float(selected_cv_df["pr_auc"].std(ddof=1)),
            "roc_auc": float(selected_cv_df["roc_auc"].std(ddof=1)),
            "brier_score": float(selected_cv_df["brier_score"].std(ddof=1)),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
        },
        {
            "stage": "final_test",
            "selected_n_estimators": selected_n_estimators,
            "selected_max_depth": selected_max_depth,
            "selected_min_samples_leaf": selected_min_samples_leaf,
            "selected_max_features": selected_max_features,
            "pr_auc": float(test_metrics["pr_auc"]),
            "roc_auc": float(test_metrics["roc_auc"]),
            "brier_score": float(test_metrics["brier_score"]),
            "rows": int(len(test_df)),
            "positives": int(y_test.sum()),
            "positive_rate": float(y_test.mean()),
            "mean_predicted_probability": float(np.mean(test_prob)),
        },
    ]
)

stage_metrics_df.to_csv(STAGE_METRICS_CSV, index=False)

total_runtime_minutes = (time.time() - search_start_time) / 60.0

print("N2 random forest run complete.")
print(f"Total runtime: {total_runtime_minutes:.2f} minutes")
print(f"Saved tuning summary to: {TUNING_SUMMARY_CSV}")
print(f"Saved selected-HP fold metrics to: {CV_FOLD_METRICS_CSV}")
print(f"Saved stage metrics to: {STAGE_METRICS_CSV}")
print(f"Saved predictions to: {PREDICTIONS_CSV}")
print()
print("Selected hyperparameters:")
print(f"  n_estimators      = {selected_n_estimators}")
print(f"  max_depth         = {selected_max_depth}")
print(f"  min_samples_leaf  = {selected_min_samples_leaf}")
print(f"  max_features      = {selected_max_features}")

Starting N2 random forest tuning search...
Total hyperparameter combinations: 108
Total fold fits in tuning stage: 648

[TUNING] Combo 1/108 | elapsed=0.0 min | n_estimators=300, max_depth=3, min_samples_leaf=5, max_features=sqrt
[TUNING] Completed combo 1/108 in 0.13 min | cv_mean_pr_auc=0.1389, cv_mean_roc_auc=0.7079, cv_mean_brier=0.04716

[TUNING] Combo 2/108 | elapsed=0.1 min | n_estimators=300, max_depth=3, min_samples_leaf=5, max_features=0.33
[TUNING] Completed combo 2/108 in 0.26 min | cv_mean_pr_auc=0.1384, cv_mean_roc_auc=0.7046, cv_mean_brier=0.04701

[TUNING] Combo 3/108 | elapsed=0.4 min | n_estimators=300, max_depth=3, min_samples_leaf=5, max_features=0.5
[TUNING] Completed combo 3/108 in 0.37 min | cv_mean_pr_auc=0.1420, cv_mean_roc_auc=0.7052, cv_mean_brier=0.04697

[TUNING] Combo 4/108 | elapsed=0.7 min | n_estimators=300, max_depth=3, min_samples_leaf=20, max_features=sqrt
[TUNING] Completed combo 4/108 in 0.12 min | cv_mean_pr_auc=0.1394, cv_mean_roc_auc=0.7080, cv_

In [1]:
# Model N3 v3: XGBoost, no winsorization
# Hyperparameters tuned by randomized search over an expanded discrete space
# using six-fold expanding-window CV through 2021.
# Thesis-standard implementation under final horse-race plan v2.
#
# IMPORTANT:
# - This script samples a fixed set of candidate configurations from the
#   expanded discrete search space using SEARCH_RANDOM_STATE.
# - Because N3 and N4 use the same parameter space, same candidate ordering,
#   and same SEARCH_RANDOM_STATE/N_RANDOM_CONFIGS, they evaluate the SAME
#   sampled hyperparameter configurations for fair comparison.

import itertools
import pandas as pd
import numpy as np
from pathlib import Path
import time
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
from xgboost import XGBClassifier

# =========================================================
# 1. USER SETTINGS
# =========================================================
BASE_DIR = Path('.')
PANEL_PATH = BASE_DIR / 'Master_Modeling_Panel_v2.csv'

SAMPLED_CONFIGS_CSV = BASE_DIR / 'Model_N3_XGB_NoWinsor_v2_Sampled_Configurations.csv'
TUNING_SUMMARY_CSV = BASE_DIR / 'Model_N3_XGB_NoWinsor_v2_Tuning_Summary.csv'
CV_FOLD_METRICS_CSV = BASE_DIR / 'Model_N3_XGB_NoWinsor_v2_SelectedHP_DevCV_Fold_Metrics.csv'
STAGE_METRICS_CSV = BASE_DIR / 'Model_N3_XGB_NoWinsor_v2_Stage_Metrics.csv'
PREDICTIONS_CSV = BASE_DIR / 'Model_N3_XGB_NoWinsor_v2_Predictions.csv'

# Expanded discrete search space (revised once after preliminary boundary-heavy runs)
N_ESTIMATORS_GRID = [200, 400, 800, 1200]
MAX_DEPTH_GRID = [2, 3, 4, 6]
LEARNING_RATE_GRID = [0.01, 0.03, 0.05, 0.10]
MIN_CHILD_WEIGHT_GRID = [1, 5, 10]
SUBSAMPLE_GRID = [0.6, 0.7, 0.85, 1.0]
COLSAMPLE_BYTREE_GRID = [0.5, 0.7, 0.85, 1.0]

# Randomized-search controls
N_RANDOM_CONFIGS = 300
SEARCH_RANDOM_STATE = 42

# Fix randomness because boosting is stochastic under subsampling
XGB_RANDOM_STATE = 42

# =========================================================
# 2. LOAD FROZEN PANEL
# =========================================================
df = pd.read_csv(PANEL_PATH, low_memory=False)

# =========================================================
# 3. LOCKED TARGET + SPLITS
# =========================================================
TARGET_COL = 'targeted_in_year'

def assign_split(year: int) -> str:
    if 2010 <= year <= 2021:
        return 'development'
    if 2022 <= year <= 2024:
        return 'final_test'
    return 'exclude'

df['split'] = df['year'].apply(assign_split)
df = df[df['split'] != 'exclude'].copy()

# =========================================================
# 4. LOCKED RAW PREDICTOR SET (SELF-CONTAINED, MATCHES LOCKED SPEC)
# =========================================================
continuous_features = [
    'roe', 'assets_to_equity', 'current_ratio', 'ebitda', 'ebitda_margin',
    'sales_to_assets', 'sales_growth', 'interest_coverage', 'net_debt_to_ebitda',
    'fcf_to_capex', 'market_cap', 'ret_3m', 'ret_6m', 'ret_1y', 'ret_2y', 'ret_3y',
    'ret_5y', 'volatility_30d', 'volatility_90d', 'volatility_180d', 'turnover_30d',
    'dividend_payout_ratio', 'buyback_yield', 'price_to_book', 'ev_to_sales',
    'ev_to_ebitda', 'tobins_q', 'fcf_yield', 'prior_campaign_count_3y',
    'prior_settlement_count_3y', 'prior_management_favorable_count_3y',
    'prior_dissident_favorable_count_3y', 'prior_campaign_count_5y',
    'prior_settlement_count_5y', 'prior_management_favorable_count_5y',
    'prior_dissident_favorable_count_5y', 'ff49_other_permno_years_in_category',
    'ret_3m_rel_peer', 'ret_6m_rel_peer', 'ret_1y_rel_peer', 'ret_2y_rel_peer',
    'ret_3y_rel_peer', 'ret_5y_rel_peer', 'log_ev_to_sales_rel_peer',
    'log_price_to_book_rel_peer', 'log_tobins_q_rel_peer', 'log_ev_to_ebitda_rel_peer',
    'ebitda_margin_rel_peer', 'sales_growth_rel_peer', 'roe_rel_peer', 'board_size',
    'board_female_ratio', 'ceo_tenure', 'top10_inst_conc_within_13f',
    'inst_ownership_pct_13f',
]

binary_features = [
    'prior_activism_3y', 'prior_activism_5y', 'history_3y_observed',
    'history_5y_observed', 'ff49_unclassified', 'classified_board', 'poison_pill',
    'dual_class', 'iss_match_found', 'rm_stale_gt_730', 'board_match_found',
    'board_stale_gt_365', 'ceo_is_female', 'ceo_chair_duality', 'ceo_match_found',
    'ceo_stale_gt_365', 'inst_match_found',
]

categorical_feature = 'ff17_for_model'
df[categorical_feature] = df['ff17_short'].fillna('Unclassified').astype(str)

ff17_categories = [
    'Cars', 'Chems', 'Clths', 'Cnstr', 'Cnsum', 'Durbl', 'FabPr', 'Finan',
    'Food', 'Machn', 'Mines', 'Oil', 'Other', 'Rtail', 'Steel', 'Trans',
    'Utils', 'Unclassified',
]

required_cols = continuous_features + binary_features + [categorical_feature, TARGET_COL, 'year', 'permno']
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f'Missing required columns: {missing_required}')

all_predictors = continuous_features + binary_features + [categorical_feature]

# =========================================================
# 5. THESIS-STANDARD METRICS
# =========================================================
def evaluate_predictions(y_true: pd.Series, y_prob: np.ndarray) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    return {
        'pr_auc': average_precision_score(y_true, y_prob),
        'roc_auc': roc_auc_score(y_true, y_prob),
        'brier_score': brier_score_loss(y_true, y_prob),
    }

# =========================================================
# 6. TRAIN-ONLY WINSORIZER
# =========================================================
class PercentileWinsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, lower=1.0, upper=99.0):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        lower_bounds = []
        upper_bounds = []
        for j in range(X.shape[1]):
            col = X[:, j]
            non_missing = col[~np.isnan(col)]
            if non_missing.size == 0:
                lower_bounds.append(np.nan)
                upper_bounds.append(np.nan)
            else:
                lower_bounds.append(np.percentile(non_missing, self.lower))
                upper_bounds.append(np.percentile(non_missing, self.upper))
        self.lower_bounds_ = np.array(lower_bounds, dtype=float)
        self.upper_bounds_ = np.array(upper_bounds, dtype=float)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float).copy()
        for j in range(X.shape[1]):
            lb = self.lower_bounds_[j]
            ub = self.upper_bounds_[j]
            if np.isnan(lb) or np.isnan(ub):
                continue
            mask = ~np.isnan(X[:, j])
            X[mask, j] = np.clip(X[mask, j], lb, ub)
        return X

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)

# =========================================================
# 7. PREPROCESSOR + MODEL PIPELINE BUILDER
# =========================================================
def build_pipeline(
    n_estimators: int,
    max_depth: int,
    learning_rate: float,
    min_child_weight: int,
    subsample: float,
    colsample_bytree: float,
):
    continuous_transformer = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median')),
        ]
    )

    binary_transformer = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='Unclassified')),
            ('onehot', OneHotEncoder(
                categories=[ff17_categories],
                drop=['Other'],
                handle_unknown='ignore',
                sparse_output=False,
            )),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ('cont', continuous_transformer, continuous_features),
            ('bin', binary_transformer, binary_features),
            ('ff17', categorical_transformer, [categorical_feature]),
        ],
        remainder='drop',
    )

    xgb_model = XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        min_child_weight=min_child_weight,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        random_state=XGB_RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
    )

    return Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('model', xgb_model),
        ]
    )

# =========================================================
# 8. FINAL SIX-FOLD EXPANDING-WINDOW CV
# =========================================================
cv_fold_defs = [
    {'fold': 'fold_1', 'train_years': list(range(2010, 2016)), 'val_years': [2016]},
    {'fold': 'fold_2', 'train_years': list(range(2010, 2017)), 'val_years': [2017]},
    {'fold': 'fold_3', 'train_years': list(range(2010, 2018)), 'val_years': [2018]},
    {'fold': 'fold_4', 'train_years': list(range(2010, 2019)), 'val_years': [2019]},
    {'fold': 'fold_5', 'train_years': list(range(2010, 2020)), 'val_years': [2020]},
    {'fold': 'fold_6', 'train_years': list(range(2010, 2021)), 'val_years': [2021]},
]

# =========================================================
# 9. REPRODUCIBLE RANDOMIZED SEARCH OVER EXPANDED SPACE
# =========================================================
all_candidate_configs = [
    {
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'learning_rate': learning_rate,
        'min_child_weight': min_child_weight,
        'subsample': subsample,
        'colsample_bytree': colsample_bytree,
    }
    for (n_estimators, max_depth, learning_rate, min_child_weight, subsample, colsample_bytree)
    in itertools.product(
        N_ESTIMATORS_GRID,
        MAX_DEPTH_GRID,
        LEARNING_RATE_GRID,
        MIN_CHILD_WEIGHT_GRID,
        SUBSAMPLE_GRID,
        COLSAMPLE_BYTREE_GRID,
    )
]

n_total_candidates = len(all_candidate_configs)
if N_RANDOM_CONFIGS > n_total_candidates:
    raise ValueError(
        f'N_RANDOM_CONFIGS={{N_RANDOM_CONFIGS}} exceeds total candidate count={{n_total_candidates}}.'
    )

rng = np.random.default_rng(SEARCH_RANDOM_STATE)
sampled_indices = rng.choice(n_total_candidates, size=N_RANDOM_CONFIGS, replace=False)
sampled_configs = [all_candidate_configs[int(i)] for i in sampled_indices]

sampled_configs_df = pd.DataFrame(sampled_configs)
sampled_configs_df.insert(0, 'sampled_config_rank', np.arange(1, len(sampled_configs_df) + 1))
sampled_configs_df.insert(1, 'original_candidate_index', sampled_indices.astype(int))
sampled_configs_df.to_csv(SAMPLED_CONFIGS_CSV, index=False)

# =========================================================
# 10. RANDOMIZED SEARCH OVER EXPANDING FOLDS
# =========================================================
tuning_rows = []
combo_counter = 0
search_start_time = time.time()

print(f"Starting N3 XGBoost randomized search...")
print(f'Total candidate combinations in expanded space: {n_total_candidates}')
print(f'Randomly sampled configurations to evaluate: {N_RANDOM_CONFIGS}')
print(f'Total fold fits in tuning stage: {N_RANDOM_CONFIGS * len(cv_fold_defs)}')
print(f'SEARCH_RANDOM_STATE: {SEARCH_RANDOM_STATE}')
print()

for cfg in sampled_configs:
    combo_counter += 1
    combo_start_time = time.time()
    elapsed_minutes = (combo_start_time - search_start_time) / 60.0

    n_estimators = int(cfg['n_estimators'])
    max_depth = int(cfg['max_depth'])
    learning_rate = float(cfg['learning_rate'])
    min_child_weight = int(cfg['min_child_weight'])
    subsample = float(cfg['subsample'])
    colsample_bytree = float(cfg['colsample_bytree'])

    print(
        f'[TUNING] Config {combo_counter}/{N_RANDOM_CONFIGS} | '
        f'elapsed={elapsed_minutes:.1f} min | '
        f'n_estimators={n_estimators}, max_depth={max_depth}, '
        f'learning_rate={learning_rate}, min_child_weight={min_child_weight}, '
        f'subsample={subsample}, colsample_bytree={colsample_bytree}'
    )

    fold_rows = []
    for fold_def in cv_fold_defs:
        fold_name = fold_def['fold']
        train_years = fold_def['train_years']
        val_years = fold_def['val_years']

        train_df = df[df['year'].isin(train_years)].copy()
        val_df = df[df['year'].isin(val_years)].copy()

        X_train = train_df[all_predictors].copy()
        y_train = train_df[TARGET_COL].copy()
        X_val = val_df[all_predictors].copy()
        y_val = val_df[TARGET_COL].copy()

        pipeline = build_pipeline(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            min_child_weight=min_child_weight,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
        )
        pipeline.fit(X_train, y_train)
        val_prob = pipeline.predict_proba(X_val)[:, 1]

        fold_metrics = evaluate_predictions(y_val, val_prob)
        fold_metrics.update(
            {
                'fold': fold_name,
                'train_year_start': min(train_years),
                'train_year_end': max(train_years),
                'val_year_start': min(val_years),
                'val_year_end': max(val_years),
                'train_rows': len(train_df),
                'train_positives': int(y_train.sum()),
                'train_positive_rate': float(y_train.mean()),
                'val_rows': len(val_df),
                'val_positives': int(y_val.sum()),
                'val_positive_rate': float(y_val.mean()),
                'mean_predicted_probability': float(np.mean(val_prob)),
            }
        )
        fold_rows.append(fold_metrics)

    fold_metrics_df = pd.DataFrame(fold_rows)
    combo_minutes = (time.time() - combo_start_time) / 60.0
    combo_mean_pr_auc = float(fold_metrics_df['pr_auc'].mean())
    combo_mean_roc_auc = float(fold_metrics_df['roc_auc'].mean())
    combo_mean_brier = float(fold_metrics_df['brier_score'].mean())

    tuning_rows.append(
        {
            'evaluated_config_rank': combo_counter,
            'n_estimators': n_estimators,
            'max_depth': max_depth,
            'learning_rate': learning_rate,
            'min_child_weight': min_child_weight,
            'subsample': subsample,
            'colsample_bytree': colsample_bytree,
            'cv_mean_pr_auc': combo_mean_pr_auc,
            'cv_std_pr_auc': float(fold_metrics_df['pr_auc'].std(ddof=1)),
            'cv_mean_roc_auc': combo_mean_roc_auc,
            'cv_std_roc_auc': float(fold_metrics_df['roc_auc'].std(ddof=1)),
            'cv_mean_brier_score': combo_mean_brier,
            'cv_std_brier_score': float(fold_metrics_df['brier_score'].std(ddof=1)),
        }
    )

    print(
        f'[TUNING] Completed config {combo_counter}/{N_RANDOM_CONFIGS} in {combo_minutes:.2f} min | '
        f'cv_mean_pr_auc={combo_mean_pr_auc:.4f}, '
        f'cv_mean_roc_auc={combo_mean_roc_auc:.4f}, '
        f'cv_mean_brier={combo_mean_brier:.5f}'
    )
    print()

tuning_summary_df = pd.DataFrame(tuning_rows)

# Preference ordering:
# 1) higher mean CV PR-AUC
# 2) higher mean CV ROC-AUC
# 3) lower mean CV Brier
# 4) simpler booster as a practical tie-break if metrics are essentially tied
#    (shallower depth, higher child weight, fewer trees, larger learning rate)
tuning_summary_df = tuning_summary_df.sort_values(
    by=[
        'cv_mean_pr_auc',
        'cv_mean_roc_auc',
        'cv_mean_brier_score',
        'max_depth',
        'min_child_weight',
        'n_estimators',
        'learning_rate',
        'subsample',
        'colsample_bytree',
    ],
    ascending=[False, False, True, True, False, True, False, True, True],
).reset_index(drop=True)

tuning_summary_df.to_csv(TUNING_SUMMARY_CSV, index=False)

selected_n_estimators = int(tuning_summary_df.loc[0, 'n_estimators'])
selected_max_depth = int(tuning_summary_df.loc[0, 'max_depth'])
selected_learning_rate = float(tuning_summary_df.loc[0, 'learning_rate'])
selected_min_child_weight = int(tuning_summary_df.loc[0, 'min_child_weight'])
selected_subsample = float(tuning_summary_df.loc[0, 'subsample'])
selected_colsample_bytree = float(tuning_summary_df.loc[0, 'colsample_bytree'])

# =========================================================
# 11. RERUN SELECTED HYPERPARAMETERS FOR FOLD METRICS
# =========================================================
selected_cv_rows = []
prediction_frames = []

for fold_def in cv_fold_defs:
    fold_name = fold_def['fold']
    train_years = fold_def['train_years']
    val_years = fold_def['val_years']

    train_df = df[df['year'].isin(train_years)].copy()
    val_df = df[df['year'].isin(val_years)].copy()

    X_train = train_df[all_predictors].copy()
    y_train = train_df[TARGET_COL].copy()
    X_val = val_df[all_predictors].copy()
    y_val = val_df[TARGET_COL].copy()

    pipeline = build_pipeline(
        n_estimators=selected_n_estimators,
        max_depth=selected_max_depth,
        learning_rate=selected_learning_rate,
        min_child_weight=selected_min_child_weight,
        subsample=selected_subsample,
        colsample_bytree=selected_colsample_bytree,
    )
    pipeline.fit(X_train, y_train)
    val_prob = pipeline.predict_proba(X_val)[:, 1]

    fold_metrics = evaluate_predictions(y_val, val_prob)
    fold_metrics.update(
        {
            'selected_n_estimators': selected_n_estimators,
            'selected_max_depth': selected_max_depth,
            'selected_learning_rate': selected_learning_rate,
            'selected_min_child_weight': selected_min_child_weight,
            'selected_subsample': selected_subsample,
            'selected_colsample_bytree': selected_colsample_bytree,
            'fold': fold_name,
            'train_year_start': min(train_years),
            'train_year_end': max(train_years),
            'val_year_start': min(val_years),
            'val_year_end': max(val_years),
            'train_rows': len(train_df),
            'train_positives': int(y_train.sum()),
            'train_positive_rate': float(y_train.mean()),
            'val_rows': len(val_df),
            'val_positives': int(y_val.sum()),
            'val_positive_rate': float(y_val.mean()),
            'mean_predicted_probability': float(np.mean(val_prob)),
        }
    )
    selected_cv_rows.append(fold_metrics)

    prediction_frames.append(
        pd.DataFrame(
            {
                'permno': val_df['permno'].to_numpy(),
                'year': val_df['year'].to_numpy(),
                'evaluation_stage': 'development_cv',
                'fold': fold_name,
                'actual': y_val.to_numpy(),
                'predicted_probability': val_prob,
                'selected_n_estimators': selected_n_estimators,
                'selected_max_depth': selected_max_depth,
                'selected_learning_rate': selected_learning_rate,
                'selected_min_child_weight': selected_min_child_weight,
                'selected_subsample': selected_subsample,
                'selected_colsample_bytree': selected_colsample_bytree,
            }
        )
    )

selected_cv_df = pd.DataFrame(selected_cv_rows)
selected_cv_df.to_csv(CV_FOLD_METRICS_CSV, index=False)

# =========================================================
# 12. FINAL LOCKED TEST: REFIT ON 2010-2021, TEST ON 2022-2024
# =========================================================
dev_df = df[df['year'].between(2010, 2021)].copy()
test_df = df[df['year'].between(2022, 2024)].copy()

X_dev = dev_df[all_predictors].copy()
y_dev = dev_df[TARGET_COL].copy()
X_test = test_df[all_predictors].copy()
y_test = test_df[TARGET_COL].copy()

final_pipeline = build_pipeline(
    n_estimators=selected_n_estimators,
    max_depth=selected_max_depth,
    learning_rate=selected_learning_rate,
    min_child_weight=selected_min_child_weight,
    subsample=selected_subsample,
    colsample_bytree=selected_colsample_bytree,
)
final_pipeline.fit(X_dev, y_dev)

test_prob = final_pipeline.predict_proba(X_test)[:, 1]
test_metrics = evaluate_predictions(y_test, test_prob)

prediction_frames.append(
    pd.DataFrame(
        {
            'permno': test_df['permno'].to_numpy(),
            'year': test_df['year'].to_numpy(),
            'evaluation_stage': 'final_test',
            'fold': 'final_test',
            'actual': y_test.to_numpy(),
            'predicted_probability': test_prob,
            'selected_n_estimators': selected_n_estimators,
            'selected_max_depth': selected_max_depth,
            'selected_learning_rate': selected_learning_rate,
            'selected_min_child_weight': selected_min_child_weight,
            'selected_subsample': selected_subsample,
            'selected_colsample_bytree': selected_colsample_bytree,
        }
    )
)

predictions_df = pd.concat(prediction_frames, ignore_index=True)
predictions_df.to_csv(PREDICTIONS_CSV, index=False)

# =========================================================
# 13. STAGE METRICS TABLE
# =========================================================
stage_metrics_df = pd.DataFrame(
    [
        {
            'stage': 'development_cv_mean',
            'selected_n_estimators': selected_n_estimators,
            'selected_max_depth': selected_max_depth,
            'selected_learning_rate': selected_learning_rate,
            'selected_min_child_weight': selected_min_child_weight,
            'selected_subsample': selected_subsample,
            'selected_colsample_bytree': selected_colsample_bytree,
            'pr_auc': float(selected_cv_df['pr_auc'].mean()),
            'roc_auc': float(selected_cv_df['roc_auc'].mean()),
            'brier_score': float(selected_cv_df['brier_score'].mean()),
            'rows': np.nan,
            'positives': np.nan,
            'positive_rate': np.nan,
            'mean_predicted_probability': np.nan,
        },
        {
            'stage': 'development_cv_std',
            'selected_n_estimators': selected_n_estimators,
            'selected_max_depth': selected_max_depth,
            'selected_learning_rate': selected_learning_rate,
            'selected_min_child_weight': selected_min_child_weight,
            'selected_subsample': selected_subsample,
            'selected_colsample_bytree': selected_colsample_bytree,
            'pr_auc': float(selected_cv_df['pr_auc'].std(ddof=1)),
            'roc_auc': float(selected_cv_df['roc_auc'].std(ddof=1)),
            'brier_score': float(selected_cv_df['brier_score'].std(ddof=1)),
            'rows': np.nan,
            'positives': np.nan,
            'positive_rate': np.nan,
            'mean_predicted_probability': np.nan,
        },
        {
            'stage': 'final_test',
            'selected_n_estimators': selected_n_estimators,
            'selected_max_depth': selected_max_depth,
            'selected_learning_rate': selected_learning_rate,
            'selected_min_child_weight': selected_min_child_weight,
            'selected_subsample': selected_subsample,
            'selected_colsample_bytree': selected_colsample_bytree,
            'pr_auc': float(test_metrics['pr_auc']),
            'roc_auc': float(test_metrics['roc_auc']),
            'brier_score': float(test_metrics['brier_score']),
            'rows': int(len(test_df)),
            'positives': int(y_test.sum()),
            'positive_rate': float(y_test.mean()),
            'mean_predicted_probability': float(np.mean(test_prob)),
        },
    ]
)

stage_metrics_df.to_csv(STAGE_METRICS_CSV, index=False)

total_runtime_minutes = (time.time() - search_start_time) / 60.0

print(f"N3 XGBoost randomized-search run complete.")
print(f'Total runtime: {total_runtime_minutes:.2f} minutes')
print(f'Saved sampled configs to: {SAMPLED_CONFIGS_CSV}')
print(f'Saved tuning summary to: {TUNING_SUMMARY_CSV}')
print(f'Saved selected-HP fold metrics to: {CV_FOLD_METRICS_CSV}')
print(f'Saved stage metrics to: {STAGE_METRICS_CSV}')
print(f'Saved predictions to: {PREDICTIONS_CSV}')
print()
print('Selected hyperparameters:')
print(f'  n_estimators      = {selected_n_estimators}')
print(f'  max_depth         = {selected_max_depth}')
print(f'  learning_rate     = {selected_learning_rate}')
print(f'  min_child_weight  = {selected_min_child_weight}')
print(f'  subsample         = {selected_subsample}')
print(f'  colsample_bytree  = {selected_colsample_bytree}')

Starting N3 XGBoost randomized search...
Total candidate combinations in expanded space: 3072
Randomly sampled configurations to evaluate: 300
Total fold fits in tuning stage: 1800
SEARCH_RANDOM_STATE: 42

[TUNING] Config 1/300 | elapsed=0.0 min | n_estimators=400, max_depth=6, learning_rate=0.03, min_child_weight=10, subsample=0.7, colsample_bytree=1.0
[TUNING] Completed config 1/300 in 0.39 min | cv_mean_pr_auc=0.1458, cv_mean_roc_auc=0.7240, cv_mean_brier=0.04685

[TUNING] Config 2/300 | elapsed=0.4 min | n_estimators=800, max_depth=6, learning_rate=0.05, min_child_weight=10, subsample=0.85, colsample_bytree=1.0
[TUNING] Completed config 2/300 in 0.72 min | cv_mean_pr_auc=0.1296, cv_mean_roc_auc=0.6900, cv_mean_brier=0.04783

[TUNING] Config 3/300 | elapsed=1.1 min | n_estimators=200, max_depth=3, learning_rate=0.1, min_child_weight=10, subsample=1.0, colsample_bytree=0.7
[TUNING] Completed config 3/300 in 0.09 min | cv_mean_pr_auc=0.1481, cv_mean_roc_auc=0.7230, cv_mean_brier=0.046

In [2]:
# Model N4 v3: XGBoost, with winsorization
# Hyperparameters tuned by randomized search over an expanded discrete space
# using six-fold expanding-window CV through 2021.
# Thesis-standard implementation under final horse-race plan v2.
#
# IMPORTANT:
# - This script samples a fixed set of candidate configurations from the
#   expanded discrete search space using SEARCH_RANDOM_STATE.
# - Because N3 and N4 use the same parameter space, same candidate ordering,
#   and same SEARCH_RANDOM_STATE/N_RANDOM_CONFIGS, they evaluate the SAME
#   sampled hyperparameter configurations for fair comparison.

import itertools
import pandas as pd
import numpy as np
from pathlib import Path
import time
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
from xgboost import XGBClassifier

# =========================================================
# 1. USER SETTINGS
# =========================================================
BASE_DIR = Path('.')
PANEL_PATH = BASE_DIR / 'Master_Modeling_Panel_v2.csv'

SAMPLED_CONFIGS_CSV = BASE_DIR / 'Model_N4_XGB_Winsor_v2_Sampled_Configurations.csv'
TUNING_SUMMARY_CSV = BASE_DIR / 'Model_N4_XGB_Winsor_v2_Tuning_Summary.csv'
CV_FOLD_METRICS_CSV = BASE_DIR / 'Model_N4_XGB_Winsor_v2_SelectedHP_DevCV_Fold_Metrics.csv'
STAGE_METRICS_CSV = BASE_DIR / 'Model_N4_XGB_Winsor_v2_Stage_Metrics.csv'
PREDICTIONS_CSV = BASE_DIR / 'Model_N4_XGB_Winsor_v2_Predictions.csv'

# Expanded discrete search space (revised once after preliminary boundary-heavy runs)
N_ESTIMATORS_GRID = [200, 400, 800, 1200]
MAX_DEPTH_GRID = [2, 3, 4, 6]
LEARNING_RATE_GRID = [0.01, 0.03, 0.05, 0.10]
MIN_CHILD_WEIGHT_GRID = [1, 5, 10]
SUBSAMPLE_GRID = [0.6, 0.7, 0.85, 1.0]
COLSAMPLE_BYTREE_GRID = [0.5, 0.7, 0.85, 1.0]

# Randomized-search controls
N_RANDOM_CONFIGS = 300
SEARCH_RANDOM_STATE = 42

# Fix randomness because boosting is stochastic under subsampling
XGB_RANDOM_STATE = 42

# =========================================================
# 2. LOAD FROZEN PANEL
# =========================================================
df = pd.read_csv(PANEL_PATH, low_memory=False)

# =========================================================
# 3. LOCKED TARGET + SPLITS
# =========================================================
TARGET_COL = 'targeted_in_year'

def assign_split(year: int) -> str:
    if 2010 <= year <= 2021:
        return 'development'
    if 2022 <= year <= 2024:
        return 'final_test'
    return 'exclude'

df['split'] = df['year'].apply(assign_split)
df = df[df['split'] != 'exclude'].copy()

# =========================================================
# 4. LOCKED RAW PREDICTOR SET (SELF-CONTAINED, MATCHES LOCKED SPEC)
# =========================================================
continuous_features = [
    'roe', 'assets_to_equity', 'current_ratio', 'ebitda', 'ebitda_margin',
    'sales_to_assets', 'sales_growth', 'interest_coverage', 'net_debt_to_ebitda',
    'fcf_to_capex', 'market_cap', 'ret_3m', 'ret_6m', 'ret_1y', 'ret_2y', 'ret_3y',
    'ret_5y', 'volatility_30d', 'volatility_90d', 'volatility_180d', 'turnover_30d',
    'dividend_payout_ratio', 'buyback_yield', 'price_to_book', 'ev_to_sales',
    'ev_to_ebitda', 'tobins_q', 'fcf_yield', 'prior_campaign_count_3y',
    'prior_settlement_count_3y', 'prior_management_favorable_count_3y',
    'prior_dissident_favorable_count_3y', 'prior_campaign_count_5y',
    'prior_settlement_count_5y', 'prior_management_favorable_count_5y',
    'prior_dissident_favorable_count_5y', 'ff49_other_permno_years_in_category',
    'ret_3m_rel_peer', 'ret_6m_rel_peer', 'ret_1y_rel_peer', 'ret_2y_rel_peer',
    'ret_3y_rel_peer', 'ret_5y_rel_peer', 'log_ev_to_sales_rel_peer',
    'log_price_to_book_rel_peer', 'log_tobins_q_rel_peer', 'log_ev_to_ebitda_rel_peer',
    'ebitda_margin_rel_peer', 'sales_growth_rel_peer', 'roe_rel_peer', 'board_size',
    'board_female_ratio', 'ceo_tenure', 'top10_inst_conc_within_13f',
    'inst_ownership_pct_13f',
]

binary_features = [
    'prior_activism_3y', 'prior_activism_5y', 'history_3y_observed',
    'history_5y_observed', 'ff49_unclassified', 'classified_board', 'poison_pill',
    'dual_class', 'iss_match_found', 'rm_stale_gt_730', 'board_match_found',
    'board_stale_gt_365', 'ceo_is_female', 'ceo_chair_duality', 'ceo_match_found',
    'ceo_stale_gt_365', 'inst_match_found',
]

categorical_feature = 'ff17_for_model'
df[categorical_feature] = df['ff17_short'].fillna('Unclassified').astype(str)

ff17_categories = [
    'Cars', 'Chems', 'Clths', 'Cnstr', 'Cnsum', 'Durbl', 'FabPr', 'Finan',
    'Food', 'Machn', 'Mines', 'Oil', 'Other', 'Rtail', 'Steel', 'Trans',
    'Utils', 'Unclassified',
]

required_cols = continuous_features + binary_features + [categorical_feature, TARGET_COL, 'year', 'permno']
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f'Missing required columns: {missing_required}')

all_predictors = continuous_features + binary_features + [categorical_feature]

# =========================================================
# 5. THESIS-STANDARD METRICS
# =========================================================
def evaluate_predictions(y_true: pd.Series, y_prob: np.ndarray) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    return {
        'pr_auc': average_precision_score(y_true, y_prob),
        'roc_auc': roc_auc_score(y_true, y_prob),
        'brier_score': brier_score_loss(y_true, y_prob),
    }

# =========================================================
# 6. TRAIN-ONLY WINSORIZER
# =========================================================
class PercentileWinsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, lower=1.0, upper=99.0):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        lower_bounds = []
        upper_bounds = []
        for j in range(X.shape[1]):
            col = X[:, j]
            non_missing = col[~np.isnan(col)]
            if non_missing.size == 0:
                lower_bounds.append(np.nan)
                upper_bounds.append(np.nan)
            else:
                lower_bounds.append(np.percentile(non_missing, self.lower))
                upper_bounds.append(np.percentile(non_missing, self.upper))
        self.lower_bounds_ = np.array(lower_bounds, dtype=float)
        self.upper_bounds_ = np.array(upper_bounds, dtype=float)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float).copy()
        for j in range(X.shape[1]):
            lb = self.lower_bounds_[j]
            ub = self.upper_bounds_[j]
            if np.isnan(lb) or np.isnan(ub):
                continue
            mask = ~np.isnan(X[:, j])
            X[mask, j] = np.clip(X[mask, j], lb, ub)
        return X

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)

# =========================================================
# 7. PREPROCESSOR + MODEL PIPELINE BUILDER
# =========================================================
def build_pipeline(
    n_estimators: int,
    max_depth: int,
    learning_rate: float,
    min_child_weight: int,
    subsample: float,
    colsample_bytree: float,
):
    continuous_transformer = Pipeline(
        steps=[
            ('winsorizer', PercentileWinsorizer(lower=1.0, upper=99.0)),
            ('imputer', SimpleImputer(strategy='median')),
        ]
    )

    binary_transformer = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='Unclassified')),
            ('onehot', OneHotEncoder(
                categories=[ff17_categories],
                drop=['Other'],
                handle_unknown='ignore',
                sparse_output=False,
            )),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ('cont', continuous_transformer, continuous_features),
            ('bin', binary_transformer, binary_features),
            ('ff17', categorical_transformer, [categorical_feature]),
        ],
        remainder='drop',
    )

    xgb_model = XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        min_child_weight=min_child_weight,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        random_state=XGB_RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
    )

    return Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('model', xgb_model),
        ]
    )

# =========================================================
# 8. FINAL SIX-FOLD EXPANDING-WINDOW CV
# =========================================================
cv_fold_defs = [
    {'fold': 'fold_1', 'train_years': list(range(2010, 2016)), 'val_years': [2016]},
    {'fold': 'fold_2', 'train_years': list(range(2010, 2017)), 'val_years': [2017]},
    {'fold': 'fold_3', 'train_years': list(range(2010, 2018)), 'val_years': [2018]},
    {'fold': 'fold_4', 'train_years': list(range(2010, 2019)), 'val_years': [2019]},
    {'fold': 'fold_5', 'train_years': list(range(2010, 2020)), 'val_years': [2020]},
    {'fold': 'fold_6', 'train_years': list(range(2010, 2021)), 'val_years': [2021]},
]

# =========================================================
# 9. REPRODUCIBLE RANDOMIZED SEARCH OVER EXPANDED SPACE
# =========================================================
all_candidate_configs = [
    {
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'learning_rate': learning_rate,
        'min_child_weight': min_child_weight,
        'subsample': subsample,
        'colsample_bytree': colsample_bytree,
    }
    for (n_estimators, max_depth, learning_rate, min_child_weight, subsample, colsample_bytree)
    in itertools.product(
        N_ESTIMATORS_GRID,
        MAX_DEPTH_GRID,
        LEARNING_RATE_GRID,
        MIN_CHILD_WEIGHT_GRID,
        SUBSAMPLE_GRID,
        COLSAMPLE_BYTREE_GRID,
    )
]

n_total_candidates = len(all_candidate_configs)
if N_RANDOM_CONFIGS > n_total_candidates:
    raise ValueError(
        f'N_RANDOM_CONFIGS={{N_RANDOM_CONFIGS}} exceeds total candidate count={{n_total_candidates}}.'
    )

rng = np.random.default_rng(SEARCH_RANDOM_STATE)
sampled_indices = rng.choice(n_total_candidates, size=N_RANDOM_CONFIGS, replace=False)
sampled_configs = [all_candidate_configs[int(i)] for i in sampled_indices]

sampled_configs_df = pd.DataFrame(sampled_configs)
sampled_configs_df.insert(0, 'sampled_config_rank', np.arange(1, len(sampled_configs_df) + 1))
sampled_configs_df.insert(1, 'original_candidate_index', sampled_indices.astype(int))
sampled_configs_df.to_csv(SAMPLED_CONFIGS_CSV, index=False)

# =========================================================
# 10. RANDOMIZED SEARCH OVER EXPANDING FOLDS
# =========================================================
tuning_rows = []
combo_counter = 0
search_start_time = time.time()

print(f"Starting N4 XGBoost randomized search...")
print(f'Total candidate combinations in expanded space: {n_total_candidates}')
print(f'Randomly sampled configurations to evaluate: {N_RANDOM_CONFIGS}')
print(f'Total fold fits in tuning stage: {N_RANDOM_CONFIGS * len(cv_fold_defs)}')
print(f'SEARCH_RANDOM_STATE: {SEARCH_RANDOM_STATE}')
print()

for cfg in sampled_configs:
    combo_counter += 1
    combo_start_time = time.time()
    elapsed_minutes = (combo_start_time - search_start_time) / 60.0

    n_estimators = int(cfg['n_estimators'])
    max_depth = int(cfg['max_depth'])
    learning_rate = float(cfg['learning_rate'])
    min_child_weight = int(cfg['min_child_weight'])
    subsample = float(cfg['subsample'])
    colsample_bytree = float(cfg['colsample_bytree'])

    print(
        f'[TUNING] Config {combo_counter}/{N_RANDOM_CONFIGS} | '
        f'elapsed={elapsed_minutes:.1f} min | '
        f'n_estimators={n_estimators}, max_depth={max_depth}, '
        f'learning_rate={learning_rate}, min_child_weight={min_child_weight}, '
        f'subsample={subsample}, colsample_bytree={colsample_bytree}'
    )

    fold_rows = []
    for fold_def in cv_fold_defs:
        fold_name = fold_def['fold']
        train_years = fold_def['train_years']
        val_years = fold_def['val_years']

        train_df = df[df['year'].isin(train_years)].copy()
        val_df = df[df['year'].isin(val_years)].copy()

        X_train = train_df[all_predictors].copy()
        y_train = train_df[TARGET_COL].copy()
        X_val = val_df[all_predictors].copy()
        y_val = val_df[TARGET_COL].copy()

        pipeline = build_pipeline(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            min_child_weight=min_child_weight,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
        )
        pipeline.fit(X_train, y_train)
        val_prob = pipeline.predict_proba(X_val)[:, 1]

        fold_metrics = evaluate_predictions(y_val, val_prob)
        fold_metrics.update(
            {
                'fold': fold_name,
                'train_year_start': min(train_years),
                'train_year_end': max(train_years),
                'val_year_start': min(val_years),
                'val_year_end': max(val_years),
                'train_rows': len(train_df),
                'train_positives': int(y_train.sum()),
                'train_positive_rate': float(y_train.mean()),
                'val_rows': len(val_df),
                'val_positives': int(y_val.sum()),
                'val_positive_rate': float(y_val.mean()),
                'mean_predicted_probability': float(np.mean(val_prob)),
            }
        )
        fold_rows.append(fold_metrics)

    fold_metrics_df = pd.DataFrame(fold_rows)
    combo_minutes = (time.time() - combo_start_time) / 60.0
    combo_mean_pr_auc = float(fold_metrics_df['pr_auc'].mean())
    combo_mean_roc_auc = float(fold_metrics_df['roc_auc'].mean())
    combo_mean_brier = float(fold_metrics_df['brier_score'].mean())

    tuning_rows.append(
        {
            'evaluated_config_rank': combo_counter,
            'n_estimators': n_estimators,
            'max_depth': max_depth,
            'learning_rate': learning_rate,
            'min_child_weight': min_child_weight,
            'subsample': subsample,
            'colsample_bytree': colsample_bytree,
            'cv_mean_pr_auc': combo_mean_pr_auc,
            'cv_std_pr_auc': float(fold_metrics_df['pr_auc'].std(ddof=1)),
            'cv_mean_roc_auc': combo_mean_roc_auc,
            'cv_std_roc_auc': float(fold_metrics_df['roc_auc'].std(ddof=1)),
            'cv_mean_brier_score': combo_mean_brier,
            'cv_std_brier_score': float(fold_metrics_df['brier_score'].std(ddof=1)),
        }
    )

    print(
        f'[TUNING] Completed config {combo_counter}/{N_RANDOM_CONFIGS} in {combo_minutes:.2f} min | '
        f'cv_mean_pr_auc={combo_mean_pr_auc:.4f}, '
        f'cv_mean_roc_auc={combo_mean_roc_auc:.4f}, '
        f'cv_mean_brier={combo_mean_brier:.5f}'
    )
    print()

tuning_summary_df = pd.DataFrame(tuning_rows)

# Preference ordering:
# 1) higher mean CV PR-AUC
# 2) higher mean CV ROC-AUC
# 3) lower mean CV Brier
# 4) simpler booster as a practical tie-break if metrics are essentially tied
#    (shallower depth, higher child weight, fewer trees, larger learning rate)
tuning_summary_df = tuning_summary_df.sort_values(
    by=[
        'cv_mean_pr_auc',
        'cv_mean_roc_auc',
        'cv_mean_brier_score',
        'max_depth',
        'min_child_weight',
        'n_estimators',
        'learning_rate',
        'subsample',
        'colsample_bytree',
    ],
    ascending=[False, False, True, True, False, True, False, True, True],
).reset_index(drop=True)

tuning_summary_df.to_csv(TUNING_SUMMARY_CSV, index=False)

selected_n_estimators = int(tuning_summary_df.loc[0, 'n_estimators'])
selected_max_depth = int(tuning_summary_df.loc[0, 'max_depth'])
selected_learning_rate = float(tuning_summary_df.loc[0, 'learning_rate'])
selected_min_child_weight = int(tuning_summary_df.loc[0, 'min_child_weight'])
selected_subsample = float(tuning_summary_df.loc[0, 'subsample'])
selected_colsample_bytree = float(tuning_summary_df.loc[0, 'colsample_bytree'])

# =========================================================
# 11. RERUN SELECTED HYPERPARAMETERS FOR FOLD METRICS
# =========================================================
selected_cv_rows = []
prediction_frames = []

for fold_def in cv_fold_defs:
    fold_name = fold_def['fold']
    train_years = fold_def['train_years']
    val_years = fold_def['val_years']

    train_df = df[df['year'].isin(train_years)].copy()
    val_df = df[df['year'].isin(val_years)].copy()

    X_train = train_df[all_predictors].copy()
    y_train = train_df[TARGET_COL].copy()
    X_val = val_df[all_predictors].copy()
    y_val = val_df[TARGET_COL].copy()

    pipeline = build_pipeline(
        n_estimators=selected_n_estimators,
        max_depth=selected_max_depth,
        learning_rate=selected_learning_rate,
        min_child_weight=selected_min_child_weight,
        subsample=selected_subsample,
        colsample_bytree=selected_colsample_bytree,
    )
    pipeline.fit(X_train, y_train)
    val_prob = pipeline.predict_proba(X_val)[:, 1]

    fold_metrics = evaluate_predictions(y_val, val_prob)
    fold_metrics.update(
        {
            'selected_n_estimators': selected_n_estimators,
            'selected_max_depth': selected_max_depth,
            'selected_learning_rate': selected_learning_rate,
            'selected_min_child_weight': selected_min_child_weight,
            'selected_subsample': selected_subsample,
            'selected_colsample_bytree': selected_colsample_bytree,
            'fold': fold_name,
            'train_year_start': min(train_years),
            'train_year_end': max(train_years),
            'val_year_start': min(val_years),
            'val_year_end': max(val_years),
            'train_rows': len(train_df),
            'train_positives': int(y_train.sum()),
            'train_positive_rate': float(y_train.mean()),
            'val_rows': len(val_df),
            'val_positives': int(y_val.sum()),
            'val_positive_rate': float(y_val.mean()),
            'mean_predicted_probability': float(np.mean(val_prob)),
        }
    )
    selected_cv_rows.append(fold_metrics)

    prediction_frames.append(
        pd.DataFrame(
            {
                'permno': val_df['permno'].to_numpy(),
                'year': val_df['year'].to_numpy(),
                'evaluation_stage': 'development_cv',
                'fold': fold_name,
                'actual': y_val.to_numpy(),
                'predicted_probability': val_prob,
                'selected_n_estimators': selected_n_estimators,
                'selected_max_depth': selected_max_depth,
                'selected_learning_rate': selected_learning_rate,
                'selected_min_child_weight': selected_min_child_weight,
                'selected_subsample': selected_subsample,
                'selected_colsample_bytree': selected_colsample_bytree,
            }
        )
    )

selected_cv_df = pd.DataFrame(selected_cv_rows)
selected_cv_df.to_csv(CV_FOLD_METRICS_CSV, index=False)

# =========================================================
# 12. FINAL LOCKED TEST: REFIT ON 2010-2021, TEST ON 2022-2024
# =========================================================
dev_df = df[df['year'].between(2010, 2021)].copy()
test_df = df[df['year'].between(2022, 2024)].copy()

X_dev = dev_df[all_predictors].copy()
y_dev = dev_df[TARGET_COL].copy()
X_test = test_df[all_predictors].copy()
y_test = test_df[TARGET_COL].copy()

final_pipeline = build_pipeline(
    n_estimators=selected_n_estimators,
    max_depth=selected_max_depth,
    learning_rate=selected_learning_rate,
    min_child_weight=selected_min_child_weight,
    subsample=selected_subsample,
    colsample_bytree=selected_colsample_bytree,
)
final_pipeline.fit(X_dev, y_dev)

test_prob = final_pipeline.predict_proba(X_test)[:, 1]
test_metrics = evaluate_predictions(y_test, test_prob)

prediction_frames.append(
    pd.DataFrame(
        {
            'permno': test_df['permno'].to_numpy(),
            'year': test_df['year'].to_numpy(),
            'evaluation_stage': 'final_test',
            'fold': 'final_test',
            'actual': y_test.to_numpy(),
            'predicted_probability': test_prob,
            'selected_n_estimators': selected_n_estimators,
            'selected_max_depth': selected_max_depth,
            'selected_learning_rate': selected_learning_rate,
            'selected_min_child_weight': selected_min_child_weight,
            'selected_subsample': selected_subsample,
            'selected_colsample_bytree': selected_colsample_bytree,
        }
    )
)

predictions_df = pd.concat(prediction_frames, ignore_index=True)
predictions_df.to_csv(PREDICTIONS_CSV, index=False)

# =========================================================
# 13. STAGE METRICS TABLE
# =========================================================
stage_metrics_df = pd.DataFrame(
    [
        {
            'stage': 'development_cv_mean',
            'selected_n_estimators': selected_n_estimators,
            'selected_max_depth': selected_max_depth,
            'selected_learning_rate': selected_learning_rate,
            'selected_min_child_weight': selected_min_child_weight,
            'selected_subsample': selected_subsample,
            'selected_colsample_bytree': selected_colsample_bytree,
            'pr_auc': float(selected_cv_df['pr_auc'].mean()),
            'roc_auc': float(selected_cv_df['roc_auc'].mean()),
            'brier_score': float(selected_cv_df['brier_score'].mean()),
            'rows': np.nan,
            'positives': np.nan,
            'positive_rate': np.nan,
            'mean_predicted_probability': np.nan,
        },
        {
            'stage': 'development_cv_std',
            'selected_n_estimators': selected_n_estimators,
            'selected_max_depth': selected_max_depth,
            'selected_learning_rate': selected_learning_rate,
            'selected_min_child_weight': selected_min_child_weight,
            'selected_subsample': selected_subsample,
            'selected_colsample_bytree': selected_colsample_bytree,
            'pr_auc': float(selected_cv_df['pr_auc'].std(ddof=1)),
            'roc_auc': float(selected_cv_df['roc_auc'].std(ddof=1)),
            'brier_score': float(selected_cv_df['brier_score'].std(ddof=1)),
            'rows': np.nan,
            'positives': np.nan,
            'positive_rate': np.nan,
            'mean_predicted_probability': np.nan,
        },
        {
            'stage': 'final_test',
            'selected_n_estimators': selected_n_estimators,
            'selected_max_depth': selected_max_depth,
            'selected_learning_rate': selected_learning_rate,
            'selected_min_child_weight': selected_min_child_weight,
            'selected_subsample': selected_subsample,
            'selected_colsample_bytree': selected_colsample_bytree,
            'pr_auc': float(test_metrics['pr_auc']),
            'roc_auc': float(test_metrics['roc_auc']),
            'brier_score': float(test_metrics['brier_score']),
            'rows': int(len(test_df)),
            'positives': int(y_test.sum()),
            'positive_rate': float(y_test.mean()),
            'mean_predicted_probability': float(np.mean(test_prob)),
        },
    ]
)

stage_metrics_df.to_csv(STAGE_METRICS_CSV, index=False)

total_runtime_minutes = (time.time() - search_start_time) / 60.0

print(f"N4 XGBoost randomized-search run complete.")
print(f'Total runtime: {total_runtime_minutes:.2f} minutes')
print(f'Saved sampled configs to: {SAMPLED_CONFIGS_CSV}')
print(f'Saved tuning summary to: {TUNING_SUMMARY_CSV}')
print(f'Saved selected-HP fold metrics to: {CV_FOLD_METRICS_CSV}')
print(f'Saved stage metrics to: {STAGE_METRICS_CSV}')
print(f'Saved predictions to: {PREDICTIONS_CSV}')
print()
print('Selected hyperparameters:')
print(f'  n_estimators      = {selected_n_estimators}')
print(f'  max_depth         = {selected_max_depth}')
print(f'  learning_rate     = {selected_learning_rate}')
print(f'  min_child_weight  = {selected_min_child_weight}')
print(f'  subsample         = {selected_subsample}')
print(f'  colsample_bytree  = {selected_colsample_bytree}')

Starting N4 XGBoost randomized search...
Total candidate combinations in expanded space: 3072
Randomly sampled configurations to evaluate: 300
Total fold fits in tuning stage: 1800
SEARCH_RANDOM_STATE: 42

[TUNING] Config 1/300 | elapsed=0.0 min | n_estimators=400, max_depth=6, learning_rate=0.03, min_child_weight=10, subsample=0.7, colsample_bytree=1.0
[TUNING] Completed config 1/300 in 0.22 min | cv_mean_pr_auc=0.1412, cv_mean_roc_auc=0.7204, cv_mean_brier=0.04695

[TUNING] Config 2/300 | elapsed=0.2 min | n_estimators=800, max_depth=6, learning_rate=0.05, min_child_weight=10, subsample=0.85, colsample_bytree=1.0
[TUNING] Completed config 2/300 in 0.49 min | cv_mean_pr_auc=0.1259, cv_mean_roc_auc=0.6869, cv_mean_brier=0.04801

[TUNING] Config 3/300 | elapsed=0.7 min | n_estimators=200, max_depth=3, learning_rate=0.1, min_child_weight=10, subsample=1.0, colsample_bytree=0.7
[TUNING] Completed config 3/300 in 0.07 min | cv_mean_pr_auc=0.1448, cv_mean_roc_auc=0.7241, cv_mean_brier=0.046